# Record experimental games

This records random legal play using the incomplete candidate engine. It does **not** train a model or identify good decks. Full Standard remains unfinished. Use notebook 09 first to inspect current readiness.

Set an explicit budget below, then run the recording cell. Each game has its own JSONL file and checksum. Capped games have no terminal reward label. Replay-only headers contain both decks and must never be used as policy inputs.


In [ ]:
from pathlib import Path
import subprocess, sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/record_candidate_games.py').exists()), None)
if ROOT is None: raise RuntimeError('Open from the death-knight-lab folder.')
EPISODES = 1
MAX_ACTIONS = 500
SEED = 71
print(f'Budget: {EPISODES} random games, at most {EPISODES * MAX_ACTIONS} actions. No learning.')


In [ ]:
command = [sys.executable, str(ROOT/'tools/record_candidate_games.py'), '--engine-root', str(ROOT/'staging/rebased-88'), '--episodes', str(EPISODES), '--max-actions', str(MAX_ACTIONS), '--seed', str(SEED), '--output', str(ROOT/'runs/candidate_trajectories')]
completed = False
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            try: item = json.loads(line)
            except ValueError:
                print(line.rstrip()); continue
            if item['event']=='progress': print(f"Game {item['episode']}/{item['total']}: {item['actions']} actions")
            elif item['event']=='saved': print(f"Saved game {item['episode']}: {'capped' if item['truncated'] else 'completed'}")
            elif item['event']=='complete':
                completed = True; print('Run manifest:', item['manifest'])
        code = process.wait()
    except BaseException:
        process.terminate(); process.wait(); raise
if code or not completed: raise RuntimeError('Recording stopped early. Only individually saved episodes are complete files.')
